In [11]:
! pip install skimage

  Using cached skimage-0.0.tar.gz (757 bytes)
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [3 lines of output]
      
      *** Please install the `scikit-image` package (instead of `skimage`) ***
      
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
ERROR: Failed to build 'skimage' when getting requirements to build wheel


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from skimage import measure

In [ ]:
def gyroid(x, y, z):
	return np.sin(x) * np.cos(y) + np.sin(y) * np.cos(z) + np.sin(z) * np.cos(x)
# Create a 3D grid of points
# Adjust the range and resolution as desired.
res = 50
x = np.linspace(-np.pi, np.pi, res)
y = np.linspace(-np.pi, np.pi, res)
z = np.linspace(-np.pi, np.pi, res)
X, Y, Z = np.meshgrid(*[np.linspace(-np.pi, np.pi, res) for _ in range(3)])
# Define the gyroid implicit function
F = gyroid(X, Y, Z)

In [ ]:


# Use marching cubes to find the isosurface for F=0
verts, faces, normals, values = measure.marching_cubes(F, level=0, spacing=(x[1]-x[0], y[1]-y[0], z[1]-z[0]))

# Convert the vertices from array indices to actual coordinates
# Marching cubes returns coordinates relative to the spacing, so we add the minimum:
verts[:, 0] += x[0]
verts[:, 1] += y[0]
verts[:, 2] += z[0]

# Plot the surface
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot the triangulated surface
mesh = ax.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2],
                       cmap='Spectral', lw=1, edgecolor='none')

# Make the plot look nice
ax.set(title='Gyroid Surface', xlabel='X', ylabel='Y', zlabel='Z')
ax.view_init(elev=30, azim=45)
fig.colorbar(mesh, ax=ax, shrink=0.5, aspect=10)
plt.tight_layout()
plt.show()


In [ ]:
# Extract the isosurface at level=0 using marching cubes
# marching_cubes returns vertices, faces, normals, and values
verts, faces, normals, values = measure.marching_cubes(F, level=0)

# The vertices returned are in the index space of the array,
# so we need to scale them to the coordinate space defined by x,y,z
# Since each axis was linearly spaced, we can convert index to coordinate:
verts_scaled = np.zeros_like(verts)
verts_scaled[:, 0] = x[0] + verts[:, 0]*(x[1]-x[0])
verts_scaled[:, 1] = y[0] + verts[:, 1]*(y[1]-y[0])
verts_scaled[:, 2] = z[0] + verts[:, 2]*(z[1]-z[0])

# Create a 3D figure
fig = plt.figure()
fig.patch.set_facecolor('black')
ax = fig.add_subplot(111, projection='3d')
# ax.grid(False)
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
	axis.pane.fill = False
	axis.pane.set_edgecolor('black')

ax.xaxis.line.set_color('red')
ax.yaxis.line.set_color('green')
ax.zaxis.line.set_color('blue')

# Create a polygon collection from the faces and vertices
mesh = Poly3DCollection(verts_scaled[faces], alpha=0.8)
mesh.set_facecolor('white')
mesh.set_edgecolor('none')
ax.add_collection3d(mesh)

# Set the plot limits
ax.set(
	facecolor='black',
	xlim=[x.min(), x.max()], ylim=[y.min(), y.max()], zlim=[z.min(), z.max()])
plt.title("Gyroid Surface")
plt.tight_layout()
plt.show()


Minimal Surface PDE:

$$
(1+u_x^2)u_{yy}-2u_xu_yu_{xy}+(1+u_y^2)u_{xx}=0
$$

If we have a surface in $\mathbb{R}^3$ that can be written as 
$$
M=\{(x,y,z)|z=u(x,y)\},
$$
then M is minimal if and only if $u(x,y)$ satisfies this PDE. This PDE ensures that the <u>mean curvature</u> of the graph is zero



Let $p$ be a point on the surface $S$ inside $\mathbb{R}^3$. Each plane through $p$ containing the normal line to $S$ cuts $S$ in a (plane) curve. Fixing a choice of unit normal gives a signed curvature to that curve. As the plane is rotated by an angle $\theta$, that curvature can vary. The maximal curvature $\kappa_1$ and minimal cuvature $\kappa_2$ are known as the <u>principal curvatures</u> of $S$.

The <u>mean curvature</u> at $p\in S$ is then the average of the signed curvature over all angles $\theta$:
$$
H = \frac{1}{2\pi}\int_0^{2\pi}\kappa(\theta)d\theta
$$

In [ ]:
def calculate_parametric_gaussian_curvature(X, Y, Z, u, v):
    Xu, Xv = np.gradient(X, u, v, edge_order=2)
    Yu, Yv = np.gradient(Y, u, v, edge_order=2)
    Zu, Zv = np.gradient(Z, u, v, edge_order=2)
    Xuu, Xuv = np.gradient(Xu, u, v, edge_order=2)
    Yuu, Yuv = np.gradient(Yu, u, v, edge_order=2)
    Zuu, Zuv = np.gradient(Zu, u, v, edge_order=2)
    Xvu, Xvv = np.gradient(Xv, u, v, edge_order=2)
    Yvu, Yvv = np.gradient(Yv, u, v, edge_order=2)
    Zvu, Zvv = np.gradient(Zv, u, v, edge_order=2)
    Xuv = (Xuv + Xvu)*0.5
    Yuv = (Yuv + Yvu)*0.5
    Zuv = (Zuv + Zvu)*0.5

    ru = np.stack((Xu, Yu, Zu), axis=-1)  # shape (num_u, num_v, 3)
    rv = np.stack((Xv, Yv, Zv), axis=-1)
    ruu = np.stack((Xuu, Yuu, Zuu), axis=-1)
    ruv = np.stack((Xuv, Yuv, Zuv), axis=-1)
    rvv = np.stack((Xvv, Yvv, Zvv), axis=-1)

    E = np.sum(ru * ru, axis=-1)  
    F = np.sum(ru * rv, axis=-1) 
    G = np.sum(rv * rv, axis=-1)  

    normal = np.cross(ru, rv)
    norm_norm = np.linalg.norm(normal, axis=-1)
    n = normal / norm_norm[..., np.newaxis]

    L = np.sum(ruu * n, axis=-1)  
    M = np.sum(ruv * n, axis=-1) 
    N = np.sum(rvv * n, axis=-1)

    # Gaussian curvature
    K = (L * N - M ** 2) / (E * G - F ** 2)
    return K

Catenoids
----
***

These are an example of minimal surfaces

In [ ]:
def catenoid_curvature(c, minval, maxval, res):
    u = np.linspace(minval, maxval, res)
    v = np.linspace(minval, maxval, res)
    U, V = np.meshgrid(u, v)
    X = c * np.cosh(V / c) * np.cos(U)
    Y = c * np.cosh(V / c) * np.sin(U)
    Z = V
    K = calculate_parametric_gaussian_curvature(X,Y,Z,u,v)
    return X, Y, Z, K

res = 100
fig = plt.figure(figsize=(12,6))
fig.suptitle("Gaussian Curvature on a Catenoid")
# fig.patch.set_facecolor('black')
ax = fig.add_subplot(131, projection='3d')
X, Y, Z, K = catenoid_curvature(1, -np.pi, np.pi, res)
norm = plt.Normalize(np.min(K), np.max(K))
colors = plt.cm.jet(norm(K))
surf = ax.plot_surface(X, Y, Z, facecolors=colors, antialiased=True, shade=False)
surf.set_edgecolor('none')
# ax.contour(X, Y, Z, levels=1000)

ax = fig.add_subplot(132, projection='3d')
X, Y, Z, K = catenoid_curvature(2, -np.pi, np.pi, res)
norm = plt.Normalize(np.min(K), np.max(K))
colors = plt.cm.jet(norm(K))
surf = ax.plot_surface(X, Y, Z, facecolors=colors, shade=False)
surf.set_edgecolor('none')
# ax.contour(X, Y, Z, levels=1000)

ax = fig.add_subplot(133, projection='3d')
X, Y, Z, K = catenoid_curvature(0.5, -np.pi, np.pi, res)
norm = plt.Normalize(np.min(K), np.max(K))
colors = plt.cm.jet(norm(K))
surf = ax.plot_surface(X, Y, Z, facecolors=colors, shade=False)
surf.set_edgecolor('none')
# ax.contour(X, Y, Z, levels=1000)

m = plt.cm.ScalarMappable(cmap='jet', norm=norm)
m.set_array(K)
fig.colorbar(m, ax=ax, shrink=0.5, aspect=10, pad=0.1)
fig.tight_layout()
plt.show()

Helicoids
----
***

These are an example of minimal surfaces

In [ ]:
def helicoid_curvature(c, minval, maxval, res):
    u = np.linspace(minval, maxval, res)
    v = np.linspace(minval, maxval, res)
    U, V = np.meshgrid(u, v)
    X = U * np.cos(c * V)
    Y = U * np.sin(c * V)
    Z = V
    K = calculate_parametric_gaussian_curvature(X,Y,Z,u,v)
    return X, Y, Z, K


res = 100
fig = plt.figure(figsize=(12,6))
# fig.patch.set_facecolor('black')
ax = fig.add_subplot(131, projection='3d')
X, Y, Z, K = helicoid_curvature(1, -np.pi, np.pi, res)
norm = plt.Normalize(np.min(K), np.max(K))
colors = plt.cm.jet(norm(K))
surf = ax.plot_surface(X, Y, Z, facecolors=colors, shade=False)
surf.set_edgecolor('none')
# ax.contour(X, Y, Z, levels=1000)

ax = fig.add_subplot(132, projection='3d')
X, Y, Z, K = helicoid_curvature(2, -np.pi, np.pi, res)
norm = plt.Normalize(np.min(K), np.max(K))
colors = plt.cm.jet(norm(K))
surf = ax.plot_surface(X, Y, Z, facecolors=colors, shade=False)
surf.set_edgecolor('none')
# ax.contour(X, Y, Z, levels=1000)

ax = fig.add_subplot(133, projection='3d')
X, Y, Z, K = helicoid_curvature(0.5, -np.pi, np.pi, res)
norm = plt.Normalize(np.min(K), np.max(K))
colors = plt.cm.jet(norm(K))
surf = ax.plot_surface(X, Y, Z, facecolors=colors, shade=False)
surf.set_edgecolor('none')
# ax.contour(X, Y, Z, levels=1000)